In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt

df = pd.read_csv("../data/sample_etf_daily_long.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

close = pd.to_numeric(df["close"])
ret = close.pct_change()
ma20 = close.rolling(20).mean()
signal = close > ma20
position = signal.shift(1).fillna(False)

# 手写版
df["strategy_ret"] = (position.astype(int) * ret).fillna(0)
df["equity_manual"] = (1 + df["strategy_ret"]).cumprod()

# vectorbt版
# 导入 vectorbt，简称 vbt。后面所有回测功能都从这里调用。
# shift(1)表示整体向下移动一行
prev_signal = signal.shift(1).fillna(False).astype(bool)
entries = signal & ~prev_signal
exits = ~signal & prev_signal

pf = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    exits=exits,
    init_cash=1.0,
    fees=0.0,
    slippage=0.0,
    freq="1D",
)

df["equity_vbt"] = pf.value()

# 对比
print((df["equity_manual"] - df["equity_vbt"]).abs().max())

2.220446049250313e-16


In [ ]:
# 自己手写一遍vectorbt的用法
close = pd.to_numeric(df["close"])
ret = close.pct_change()
ma20 = close.rolling(20).mean()
signal = close > ma20
position = signal.shift(1).fillna(False)
# signal position entry entry(s1) 
# false   false   false   false
# true    false   true    false
# true    true    false   true
entries = (signal & ~position).shift(1).fillna(False).astype(bool)
# signal position exit  exit(s1)
# true    true    false   false
# false   true    true    false
# false   false   false   true
exits = (~signal & position).shift(1).fillna(False).astype(bool)

# fee 交易费用
# slippage 滑点
# freq 频率
pf = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    exits=exits,
    init_cash=1.0,
    fees=0.0,
    slippage=0.0,
    freq="1D"
)

df["equity_vbt"] = pf.value()
df["equity_vbt"]
